# 05 — Uplift Modeling: Determining Who to Target
### SmartTel Retention Intelligence — (Future Work / Experimental Extension)

**Input:** `telco_features_scored_latest.parquet` (produced by `03_classification_benchmarking.ipynb`)

>  The core system (Modules 1–3, 6–8) is complete and deployable without this module. Everything below runs on a **simulated** retention pilot, because the source dataset contains no retention-offer data. The notebook is built to answer three practical questions rather than to show that uplift modeling "wins":
> 1. Can standard uplift learners recover a known response pattern from a pilot of this size?
> 2. Under what conditions does uplift targeting beat the churn-risk targeting we already have from Module 3?
> 3. How large would a real randomized pilot need to be?

**Scope of this notebook** :
1. Design a semi-synthetic randomized pilot with an explicit, documented, planted response structure
2. Fit T-learner and X-learner uplift models and evaluate them with Qini curves and ground-truth recovery
3. Compare uplift targeting against churn-risk targeting, including a sweep over how strongly the two diverge
4. Estimate how pilot size changes the answer
5. Classify customers into **Persuadables, Sure Things, Lost Causes, Sleeping Dogs**, and validate those labels against ground truth
6. Preview the targeting trade-off (customers saved vs. backfires) as a function of campaign budget
7. Stress-test the pipeline when offer assignment is *not* randomized
8. Persist clearly-labeled experimental outputs


## What This Module Can and Cannot Claim

**Why a simulation is unavoidable.** Uplift modeling needs data in which some customers received a retention offer and others did not, with assignment that is random (or at least free of hidden bias), and churn observed afterward. Every `Churn` label in the IBM Telco dataset was observed under "no recorded offer". Pretending `Churn` is the outcome of a randomly assigned synthetic offer would embed **zero** treatment effect (the offer would be independent of the outcome by construction), so any model fit to it would simply learn noise.

**What this notebook does instead (a semi-synthetic design).** It keeps the real customers and their real covariates, uses Module 3's calibrated churn probability as each customer's no-offer churn risk, and then **plants an explicit, documented response structure** on top. It simulates a randomized pilot from that structure and asks whether standard uplift methods can recover it, and whether acting on it would beat the churn-risk targeting that Modules 1–3 already support.

**What it can claim**
- The uplift pipeline (simulate → fit → evaluate → segment → decide) works end to end and is ready to be pointed at real pilot data
- How well T- and X-learners recover a *known* effect at roughly this sample size
- Under which conditions uplift targeting beats risk targeting, and under which it does not
- A planning estimate of how large a real pilot would need to be

**What it cannot claim**
- Anything about how real SmartTel customers respond to real offers. Every response effect here is an assumption written by the analyst, so any "uplift wins" result in a given scenario is true **by construction**, and is reported next to scenarios where it is not
- That the four-segment labels are reliable for real customers. They are validated here only against a simulation we control

**Uplift sign convention used throughout:** *uplift = reduction in churn probability if the customer is contacted* (equivalently, the increase in retention probability). Positive values mean the offer helps; negative values mean it backfires.


## 1. Setup & Imports

In [1]:
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import date

from scipy.optimize import brentq
from scipy.stats import spearmanr
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression, RidgeCV
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBClassifier, XGBRegressor

from sklift.metrics import qini_auc_score, qini_curve, uplift_at_k

# scikit-uplift 0.5.1 calls a scikit-learn helper that is deprecated in scikit-learn 1.8.
# The warning is harmless; it is silenced only to keep the notebook output readable.
warnings.filterwarnings("ignore", message=".*stable_cumsum.*")

pd.set_option("display.max_columns", 100)
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (9, 5)

FEATURE_STORE_DIR = Path("../data/feature_store")
CONFIGS_DIR = Path("../configs")
CONFIGS_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)


## 2. Load Scored Dataset

In [4]:
df = pd.read_parquet(FEATURE_STORE_DIR / "telco_features_scored_latest.parquet")
n = len(df)
print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Mean calibrated churn probability: {df['churn_probability'].mean():.4f}")
df[["customerID", "Contract", "tenure", "MonthlyCharges", "persona_name", "churn_probability"]].head(15)


Shape: 7,043 rows x 29 columns
Mean calibrated churn probability: 0.2665


,customerID,Contract,tenure,MonthlyCharges,persona_name,churn_probability
0,7590-VHVEG,Month-to-month,1,29.85,New & Price-Sensitive,0.640019
1,5575-GNVDE,One year,34,56.95,New & Price-Sensitive,0.054867
2,3668-QPYBK,Month-to-month,2,53.85,New & Price-Sensitive,0.405564
3,7795-CFOCW,One year,45,42.30,New & Price-Sensitive,0.059328
4,9237-HQITU,Month-to-month,2,70.70,New & Price-Sensitive,0.574892
5,9305-CDSKC,Month-to-month,8,99.65,New & High-Risk,0.961530
6,1452-KIOVK,Month-to-month,22,89.10,New & High-Risk,0.449048
7,6713-OKOMC,Month-to-month,10,29.75,New & Price-Sensitive,0.145913
8,7892-POOKP,Month-to-month,28,104.80,New & High-Risk,0.626501
9,6388-TABGU,One year,62,56.15,New & Price-Sensitive,0.022516
